# M6 · ①·⑥·⑧ 검색 품질 비교

기존 지침 10문항에 SIF 사례 8문항과 고용노동부 보고서 3문항을 더해 정답 문서 ID를 고정했다. 같은 21문항에서 문자·의미·혼합 검색을 비교한다. Chroma에 저장된 전체 벡터를 정확한 코사인 점수로 정렬한다.

## 평가 질문·정답 고정

In [ ]:
from pathlib import Path
import hashlib,json
import pandas as pd
from IPython.display import display
from src.retrieval_eval import evaluate
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
questions=pd.DataFrame(json.loads('''[{"qid": "Q01", "query": "포크리프트 하역할 때 주의할 점", "gold_doc_ids": "B-M-11-2025", "source_type": "kosha_guide", "split": "original_guide"}, {"qid": "Q02", "query": "모바일크레인 작업할 때 위험 요소", "gold_doc_ids": "B-M-8-2025", "source_type": "kosha_guide", "split": "original_guide"}, {"qid": "Q03", "query": "백호 작업 중 사고를 막는 방법", "gold_doc_ids": "D-C-4-2025", "source_type": "kosha_guide", "split": "original_guide"}, {"qid": "Q04", "query": "사다리에서 높은 곳 작업 시 안전수칙", "gold_doc_ids": "A-G-4-2025", "source_type": "kosha_guide", "split": "original_guide"}, {"qid": "Q05", "query": "용접 중 불이 나지 않게 하려면", "gold_doc_ids": "A-G-14-2026", "source_type": "kosha_guide", "split": "original_guide"}, {"qid": "Q06", "query": "비계 조립과 작업 안전", "gold_doc_ids": "D-C-7-2026", "source_type": "kosha_guide", "split": "original_guide"}, {"qid": "Q07", "query": "전기가 흐르는 선로 근처에서 작업", "gold_doc_ids": "B-E-11-2026", "source_type": "kosha_guide", "split": "original_guide"}, {"qid": "Q08", "query": "작업발판이 달린 거푸집 안전", "gold_doc_ids": "D-C-8-2026", "source_type": "kosha_guide", "split": "original_guide"}, {"qid": "Q09", "query": "고소작업대 선정과 관리 방법", "gold_doc_ids": "M-155-2023", "source_type": "kosha_guide", "split": "original_guide"}, {"qid": "Q10", "query": "크레인에 사용하는 줄걸이 와이어로프", "gold_doc_ids": "B-M-12-2025", "source_type": "kosha_guide", "split": "original_guide"}, {"qid": "S01", "query": "제품 적재 중 화물용 리프트와 작업장 사이 개구부로 떨어진 사고 사례", "gold_doc_ids": "SIF-M-0001", "source_type": "sif", "split": "new_multi_source"}, {"qid": "S02", "query": "섬유공장 송풍기 수배전반 3.3kV 점검 중 충전부 접촉 감전 사례", "gold_doc_ids": "SIF-M-0530", "source_type": "sif", "split": "new_multi_source"}, {"qid": "S03", "query": "화학물질이 남아 있던 드럼통을 산소절단기로 자르다 폭발한 사례", "gold_doc_ids": "SIF-M-0150", "source_type": "sif", "split": "new_multi_source"}, {"qid": "S04", "query": "굴착기 버킷 안 쓰레기를 치우다 버킷과 집게 사이에 끼인 사례", "gold_doc_ids": "SIF-M-0505", "source_type": "sif", "split": "new_multi_source"}, {"qid": "S05", "query": "지게차 포크의 파렛트 위에서 덕트를 해체하다 함께 떨어진 사고", "gold_doc_ids": "SIF-M-0135", "source_type": "sif", "split": "new_multi_source"}, {"qid": "S06", "query": "알루미늄 분진 집진기를 점검하다 정전기 추정 점화원으로 폭발한 사례", "gold_doc_ids": "SIF-M-0303", "source_type": "sif", "split": "new_multi_source"}, {"qid": "S07", "query": "철근 운반 중 화물용 리프트 구간 받침 각재가 부러져 추락한 건설현장 사례", "gold_doc_ids": "SIF-C-0465", "source_type": "sif", "split": "new_multi_source"}, {"qid": "S08", "query": "자동저장창고 청소 중 움직이는 스태커 크레인에 부딪힌 사고", "gold_doc_ids": "SIF-M-0046", "source_type": "sif", "split": "new_multi_source"}, {"qid": "R01", "query": "달서구 파지 화물차 덮개를 벗기고 내려오다 떨어진 사고 조사보고서", "gold_doc_ids": "MOEL-20260501030-01", "source_type": "moel_report", "split": "new_multi_source"}, {"qid": "R02", "query": "달성군 파고라 도장 중 이동식비계에서 1.8m 추락한 사고 조사보고서", "gold_doc_ids": "MOEL-20260501028-01", "source_type": "moel_report", "split": "new_multi_source"}, {"qid": "R03", "query": "화성시 재생플라스틱 사업장에서 찢어진 톤백을 보수하다 깔린 사고 조사보고서", "gold_doc_ids": "MOEL-20260501026-01", "source_type": "moel_report", "split": "new_multi_source"}]'''))
path=ROOT/'data'/'personal'/'evaluation'/'questions_multisource.csv'
path.parent.mkdir(parents=True,exist_ok=True)
questions.to_csv(path,index=False,encoding='utf-8-sig')
assert len(questions)==21 and questions.qid.is_unique
question_sha=hashlib.sha256(path.read_bytes()).hexdigest()
assert question_sha=='a3c203f222c24c7dfed74ba5486cde3faa181558e28eb4e7433a4509035872a0'
display(questions.groupby('source_type').size().rename('문항수'))
print('질문 SHA-256:',question_sha)

## 동일 문항으로 검색 방식 비교

In [ ]:
details,summary=evaluate(ROOT)
display(summary.sort_values(['scope','source_type','index']).round(3))
display(details[(details.scope=='source_filtered')&(details.hit5==0)][['qid','source_type','index','gold_doc_ids','retrieved_doc_ids']])

## 자료 유형별 결과 그림

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import Image
if any(font.name=='Malgun Gothic' for font in font_manager.fontManager.ttflist):
    plt.rcParams['font.family']='Malgun Gothic'
plt.rcParams['axes.unicode_minus']=False
figure_data=summary[(summary.scope=='source_filtered')&(summary.source_type!='ALL')]
wide=figure_data.pivot(index='source_type',columns='index',values='hit5').reindex(['kosha_guide','sif','moel_report'])
ax=wide.plot(kind='bar',figsize=(9,5),ylim=(0,1.08),rot=0,color=['#3e667d','#9a6d55','#56917b'])
ax.set(xlabel='자료 유형',ylabel='Hit@5',title='21문항 · 자료 유형 필터 · 문서 단위 Hit@5')
for bars in ax.containers:ax.bar_label(bars,fmt='%.2f',padding=2)
fig=ax.get_figure();fig.tight_layout()
out=ROOT/'data'/'personal'/'figures'/'rag_multisource_search_comparison.png'
out.parent.mkdir(parents=True,exist_ok=True)
fig.savefig(out,dpi=170,bbox_inches='tight');plt.close(fig)
display(Image(filename=str(out)))

## 다자료 검색 평가 결과

- 고정 질문 21개는 KOSHA GUIDE 10, SIF 사례 8, 고용노동부 사고조사보고서 3개다. 정답 문서 ID가 상위 5개 서로 다른 문서 안에 있는지로 평가했다.
- 전체 자료 의미 검색 기준선은 Hit@5 16/21(0.762), Precision@5 0.152, MRR@5 0.613이었다. 자료 종류를 질문에 맞춰 미리 필터링한 경우 점수도 따로 산출해 전체 자료 검색과 구분했다.
- 질문별 정답 라벨은 검색 결과를 보기 전에 고정해 평가 누출을 줄였다. 그래도 21개는 작은 테스트셋이라 배포 후 질문 분포를 대표한다고 볼 수 없다.
- 검색 지표는 정답 문서를 찾는 순위만 본다. 생성 답변 정확도나 사고예방 효과는 측정하지 않는다.

### 재현할 때 주의할 점

- 질문 CSV, 정답 문서 ID, 코퍼스 해시가 달라지면 이전 점수와 직접 비교하지 않는다.
- 한 사례에서 여러 청크가 나와도 고유 문서 단위로 중복을 제거한 뒤 top-5를 계산한다.
- 자료 종류 필터는 실제 사용자가 해당 유형을 알고 있다는 가정이다. 전체 자료 검색 점수도 함께 보고한다.
